# ⭐ FaceStats — Attractiveness Model Inference Notebook (fs05)

This notebook loads:

- The trained **attractiveness regressor**
- Precomputed **CLIP ViT-H/14 LAION2B embeddings**
- Preprocessed face images (512×512)

It provides:

### ✅ Single-image prediction  
### ✅ Batch prediction  
### ✅ Global attractiveness ranking  
### ✅ Image previews with score  
### ✅ PCA visualization for interpretability  
### ✅ Utilities to explore results interactively  

This notebook assumes the following pipeline has already been run:

1. **fs01_preprocess_all.py** → 512×512 processed images  
2. **fs02_feature_extraction.ipynb** → preprocess pretrained features  
3. **fs03_embedding_extraction.ipynb** → CLIP ViT-H/14 embeddings  
4. **fs04_attractiveness_model.ipynb** → trained regressor saved to `models/`  

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6,6)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DEVICE


'mps'

## 🧠 Load Attractiveness Regression Model

We reconstruct the model architecture exactly the same as training so the
saved `state_dict` can be loaded correctly.

In [2]:
class AttractivenessRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1280, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.25),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.15),

            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


## 📦 Load Trained Model

We load the regressor from:
- ```models/attractiveness_regressor.pt```

In [3]:
import pickle
import os

sample_path = "dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000001.pickle"

print("Inspecting:", sample_path)

with open(sample_path, "rb") as f:
    obj = pickle.load(f)

print("Type:", type(obj))
print("Keys or length:", getattr(obj, "keys", lambda: None)(), getattr(obj, "__len__", lambda: None)())
obj


Inspecting: dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000001.pickle
Type: <class 'dict'>
Keys or length: dict_keys(['CLIP_ViTL_14@336', 'CLIP_ResNet50x64', 'CLIP_ViT_H_14_LAION2B', 'ConvNext_XL_Imagenet21k']) 4


{'CLIP_ViTL_14@336': array([[-2.3730e-01,  4.9902e-01, -4.4727e-01,  4.7314e-01,  1.6760e-01,
         -1.3062e-01,  4.2139e-01,  1.0439e+00,  2.8656e-02, -1.5125e-01,
          8.7128e-03, -4.2480e-01, -1.3416e-01, -2.1045e-01,  5.4291e-02,
          2.4536e-02, -3.7402e-01, -6.8481e-02, -1.3342e-01, -5.8594e-02,
          1.7065e-01, -4.5630e-01, -6.9238e-01, -2.2351e-01, -9.5898e-01,
         -4.0381e-01,  1.4404e-01, -1.9324e-01,  4.4678e-01,  6.2752e-03,
         -5.2246e-01,  5.2197e-01, -2.1851e-01,  1.8237e-01,  2.8687e-01,
          1.7664e-01, -4.2676e-01,  8.6670e-02, -2.5122e-01, -5.6104e-01,
          1.3464e-01,  2.5415e-01, -3.0127e-01,  4.8004e-02, -1.3831e-01,
         -2.9370e-01, -6.8408e-01, -4.3549e-02, -7.4097e-02, -5.2930e-01,
          9.5410e-01, -6.2109e-01,  1.6797e-01,  2.5732e-01,  6.4209e-01,
         -2.2253e-01,  5.1758e-02, -4.6826e-01, -2.5830e-01, -6.5918e-01,
         -8.0139e-02, -6.3660e-02,  2.8174e-01,  2.9224e-01,  9.4727e-01,
         -1.3416e-

## 📊 Load CLIP Embeddings and Feature Index

We load:

- `embeddings/features.npy` → shape `(N, 1280)`
- `embeddings/feature_index.json` → list of `{id: "...", path: "..."}`

In [4]:
EMBED_PATH = "embeddings/features.npy"
INDEX_PATH = "embeddings/feature_index.json"

X = np.load(EMBED_PATH)
with open(INDEX_PATH, "r") as f:
    feat_index = json.load(f)

print("Embedding matrix:", X.shape)
print("Index entries:", len(feat_index))

Embedding matrix: (125754, 1280)
Index entries: 125754


## 🔍 Build image ID lookup tables

This allows retrieving:
- embedding index from image ID  
- image ID from embedding index

In [5]:
# Build lookup dictionaries
id_to_idx = {entry["id"]: idx for idx, entry in enumerate(feat_index)}
idx_to_id = {idx: entry["id"] for idx, entry in enumerate(feat_index)}

# Preview
list(id_to_idx.items())[:5]

[('SFHQ_pt4_00000001', 0),
 ('SFHQ_pt4_00000002', 1),
 ('SFHQ_pt4_00000003', 2),
 ('SFHQ_pt4_00000004', 3),
 ('SFHQ_pt4_00000005', 4)]

## ⭐ Single Image Prediction

This function takes an image ID (e.g., `"SFHQ_pt4_00012345"`) and:

1. Finds its embedding row  
2. Runs the regressor  
3. Returns the predicted attractiveness score (float)  

This is the core utility for all downstream ranking and visualization steps.

In [6]:
# --- Safety check: ensure the model is loaded ---
try:
    model
except NameError:
    print("Model not found in memory → loading now...")

    model_path = "models/attractiveness_regressor.pt"
    model = AttractivenessRegressor().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    print("Loaded model from:", model_path)


def predict_score(image_id):
    """
    Predict attractiveness for a single image ID using the trained model.
    """
    if image_id not in id_to_idx:
        raise ValueError(f"Unknown image ID: {image_id}")

    idx = id_to_idx[image_id]

    # Extract embedding
    emb = torch.tensor(X[idx], dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(emb).cpu().numpy().flatten()[0]

    return float(pred)


# Demo
example_id = feat_index[0]["id"]
print("Testing:", example_id)
print("Predicted attractiveness:", predict_score(example_id))


Model not found in memory → loading now...
Loaded model from: models/attractiveness_regressor.pt
Testing: SFHQ_pt4_00000001
Predicted attractiveness: 4.0091938972473145


## Compute predictions for all embeddings

In [8]:
# Compute attractiveness predictions for all portraits
import numpy as np
import torch

print("Computing predictions for all embeddings...")

with torch.no_grad():
    emb_tensor = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    preds = model(emb_tensor).cpu().numpy().flatten()

print("Done.")
print("Preds shape:", preds.shape)
print("Min / Max:", preds.min(), preds.max())


Computing predictions for all embeddings...
Done.
Preds shape: (125754,)
Min / Max: 0.8403608 10.771513


## Save Predictions to Disk

In [9]:
import numpy as np
import os

os.makedirs("embeddings", exist_ok=True)

PRED_PATH = "embeddings/predictions.npy"
np.save(PRED_PATH, preds)

print("Saved predictions to:", PRED_PATH)
print("Shape:", preds.shape, "min/max:", preds.min(), preds.max())

Saved predictions to: embeddings/predictions.npy
Shape: (125754,) min/max: 0.8403608 10.771513
